# Customer Behavior & Sales Insights

## Project overview
Analyze the Brazilian E-Commerce Public Dataset by Olist to explore product revenue, customer repeat-purchase behavior, payment methods, delivery performance, and review scores.

## Tools
Python, pandas, NumPy, Matplotlib, and Seaborn.

## Dataset
Download the Olist Brazilian E-Commerce dataset from Kaggle and place the CSV files listed in the setup instructions in one folder. The notebook does not include the raw data.

## How to run
1. Install packages from `requirements.txt`.
2. Download and extract the Olist dataset.
3. Set `DATA_DIR` in the dataset-loading cell to the folder containing the CSV files.
4. Run all cells from top to bottom.

**Metric note:** product revenue is calculated from item `price`; freight is kept separate. The main sales analysis filters to delivered orders.


# **Step 1**

#  Import libraries

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Chart settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries imported successfully!")


Libraries imported successfully!


# Connect to dataset

In [ ]:
# Update this path to the folder containing the downloaded Olist CSV files.
# Kaggle example: Path("/kaggle/input/olistbr/brazilian-ecommerce")
DATA_DIR = Path("data/olist")

if not DATA_DIR.exists():
    kaggle_path = Path("/kaggle/input/datasets/olistbr/brazilian-ecommerce")
    if kaggle_path.exists():
        DATA_DIR = kaggle_path

required_files = [
    "olist_orders_dataset.csv", "olist_order_items_dataset.csv",
    "olist_customers_dataset.csv", "olist_products_dataset.csv",
    "olist_order_payments_dataset.csv", "olist_order_reviews_dataset.csv",
    "olist_sellers_dataset.csv", "product_category_name_translation.csv"
]
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
print("Dataset directory:", DATA_DIR.resolve())
if missing_files:
    print("Missing CSV files:", missing_files)
    print("Download the Olist dataset and set DATA_DIR to the extracted folder.")
else:
    print("All required CSV files found.")

# Load the datasets

In [ ]:

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    DATA_DIR / "product_category_name_translation.csv"
)

print("All datasets loaded successfully!")


# Check the dataset sizes

In [ ]:

datasets = {
    "Orders": orders,
    "Order Items": items,
    "Customers": customers,
    "Products": products,
    "Payments": payments,
    "Reviews": reviews,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")


# **Step 2** 

# Preview the datasets

In [ ]:

print("ORDERS DATA")
display(orders.head())

print("ORDER ITEMS DATA")
display(items.head())

print("CUSTOMERS DATA")
display(customers.head())

print("PRODUCTS DATA")
display(products.head())


This helps you understand what each dataset contains.


# Check missing values

In [ ]:

for name, df in datasets.items():
    print(f"\n--- {name} ---")

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if missing.empty:
        print("No missing values found.")
    else:
        print(missing.sort_values(ascending=False))


*Important: Missing values are not always errors. For example, an order that was cancelled may not have a delivery date. We'll handle these carefully instead of deleting rows blindly.*

# Check duplicate rows

In [ ]:

for name, df in datasets.items():
    print(
        f"{name}: {df.duplicated().sum():,} duplicate rows"
    )


*Don't remove duplicates automatically yet. We'll check the relevant IDs before deciding what to remove.*

# Convert order dates

Order dates are currently likely stored as text. Convert them to proper datetime values so we can analyze monthly sales and delivery times.


In [ ]:

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

print("Date conversion completed!")

display(orders[date_columns].dtypes)


# Check important identifiers

In [ ]:

print("Unique order IDs:", orders["order_id"].nunique())
print("Unique customer IDs:", customers["customer_id"].nunique())
print("Unique customer identities:", customers["customer_unique_id"].nunique())
print("Unique product IDs:", products["product_id"].nunique())

print("\nOrder status counts:")
display(orders["order_status"].value_counts())


These checks help confirm that our identifiers are suitable for merging the datasets later.


# **Step 3: Create the sales dataset**

# Merge orders and order items

In [ ]:

# Start with a fresh sales dataset
sales = items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Sales dataset created successfully!")
print("Rows:", sales.shape[0])
print("Columns:", sales.shape[1])

display(sales.head())


# Keep delivered orders
For our main sales analysis, we'll use delivered orders with valid purchase dates.

In [ ]:

sales = sales[
    (sales["order_status"] == "delivered") &
    (sales["order_purchase_timestamp"].notna())
].copy()

print("Delivered sales records:", len(sales))
print("Unique delivered orders:", sales["order_id"].nunique())
print("Total item revenue:", round(sales["price"].sum(), 2))


> Note: We use price for product revenue. Freight charges are separate, so we won't accidentally mix them into product revenue.

# Calculate monthly sales

In [ ]:

sales["order_month"] = (
    sales["order_purchase_timestamp"].dt.to_period("M")
)

monthly_sales = (
    sales.groupby("order_month", as_index=False)
    .agg(
        revenue=("price", "sum"),
        total_orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count")
    )
)

monthly_sales["order_month"] = (
    monthly_sales["order_month"].astype(str)
)

display(monthly_sales.head(10))


# Visualize monthly revenue

In [ ]:

plt.figure(figsize=(14, 6))

sns.lineplot(
    data=monthly_sales,
    x="order_month",
    y="revenue",
    marker="o"
)

plt.title("Monthly Sales Revenue")
plt.xlabel("Month")
plt.ylabel("Product Revenue")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


**This chart will show how product revenue changes over time.**

# Sales KPI summary

In [ ]:

total_revenue = sales["price"].sum()
total_orders = sales["order_id"].nunique()
total_items = sales["order_item_id"].count()
average_order_value = total_revenue / total_orders if total_orders else 0

print("===== SALES KPI SUMMARY =====")
print(f"Total Product Revenue: {total_revenue:,.2f}")
print(f"Total Delivered Orders: {total_orders:,}")
print(f"Total Items Sold: {total_items:,}")
print(f"Average Order Value: {average_order_value:,.2f}")


*Average order value here means product revenue divided by the number of unique delivered orders.
*

 # **Step 4: Product Category Analysis**

# Add product category names

In [ ]:

print("Translation columns:")
print(category_translation.columns.tolist())

print("\nSales category columns:")
print([
    col for col in sales.columns
    if "categor" in col.lower()
])


**Instead of merging again into your existing sales DataFrame, create a fresh category lookup and merge it into a new DataFrame. Run this complete cell:**

In [ ]:

# Create a clean product-category lookup
category_lookup = products[
    ["product_id", "product_category_name"]
].merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

# Rename the English category column consistently
category_lookup = category_lookup.rename(
    columns={
        "product_category_name_english": "category_english"
    }
)

# If the English column is unavailable, use the original category
if "category_english" not in category_lookup.columns:
    category_lookup["category_english"] = (
        category_lookup["product_category_name"]
    )

category_lookup["category_english"] = (
    category_lookup["category_english"]
    .fillna(category_lookup["product_category_name"])
    .fillna("Unknown")
)

# Remove any category columns from previous attempts
sales_base = sales.drop(
    columns=[
        "product_category_name",
        "product_category_name_english",
        "category_english"
    ],
    errors="ignore"
)

# Add category information once
sales_category = sales_base.merge(
    category_lookup[
        ["product_id", "category_english"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

sales_category["category_english"] = (
    sales_category["category_english"].fillna("Unknown")
)

print("Category merge successful!")
display(
    sales_category[
        ["product_id", "category_english", "price"]
    ].head()
)


In [ ]:

# Add product information
sales = sales.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# Add English category names
sales = sales.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

# Use the English name where available
sales["category_english"] = (
    sales["product_category_name_english"]
    .fillna(sales["product_category_name"])
    .fillna("Unknown")
)

print("Product categories added successfully!")
display(sales[["product_id", "category_english", "price"]].head())


In [ ]:

category_sales = (
    sales_category.groupby("category_english", as_index=False)
    .agg(
        revenue=("price", "sum"),
        items_sold=("order_item_id", "count"),
        total_orders=("order_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

display(category_sales.head(10))


In [ ]:

top_10_categories = category_sales.head(10)

plt.figure(figsize=(12, 7))
sns.barplot(
    data=top_10_categories,
    x="revenue",
    y="category_english"
)

plt.title("Top 10 Product Categories by Revenue")
plt.xlabel("Product Revenue")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()


Important: We are using sales_category instead of modifying sales again. This helps prevent duplicate-column errors in later steps.


# Add customer identifiers safely

In [ ]:

# Create a clean customer lookup
customer_lookup = customers[
    ["customer_id", "customer_unique_id"]
].drop_duplicates(subset=["customer_id"])

# Use the sales data from the category analysis
customer_sales = sales_category.drop(
    columns=["customer_unique_id"],
    errors="ignore"
).merge(
    orders[["order_id", "customer_id"]].drop_duplicates(
        subset=["order_id"]
    ),
    on="order_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_order")
)

# If customer_id already existed, use the original column
if "customer_id_order" in customer_sales.columns:
    customer_sales["customer_id"] = (
        customer_sales["customer_id_order"]
        .fillna(customer_sales["customer_id"])
    )
    customer_sales = customer_sales.drop(
        columns=["customer_id_order"]
    )

customer_sales = customer_sales.merge(
    customer_lookup,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Customer identifiers added!")
print("Unique customer identities:", customer_sales["customer_unique_id"].nunique())

display(
    customer_sales[
        ["order_id", "customer_id", "customer_unique_id", "price"]
    ].head()
)


# Count orders per customer
Here, a repeat customer is someone who placed more than one distinct order.

In [ ]:

customer_orders = (
    customer_sales.groupby("customer_unique_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_spend=("price", "sum")
    )
)

customer_orders["customer_type"] = np.where(
    customer_orders["total_orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

display(customer_orders.head())


# Summarize one-time and repeat customers

In [ ]:

customer_summary = (
    customer_orders.groupby("customer_type", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        average_spend_per_customer=("total_spend", "mean")
    )
)

customer_summary["percentage"] = (
    customer_summary["customers"]
    / customer_summary["customers"].sum()
    * 100
)

display(customer_summary)


# Visualize customer types

In [ ]:

plt.figure(figsize=(8, 5))

sns.barplot(
    data=customer_summary,
    x="customer_type",
    y="percentage"
)

plt.title("One-Time vs Repeat Customers")
plt.xlabel("Customer Type")
plt.ylabel("Percentage of Customers")
plt.tight_layout()
plt.show()


# Find the highest-spending customers

In [ ]:

top_customers = customer_orders.sort_values(
    "total_spend",
    ascending=False
).head(10)

display(top_customers)


What this analysis tells you
* Customer retention: The share of customers who ordered more than once.
* **Spending behavior**: Average total product spend per customer type.
* **High-value customers**: The customers with the highest recorded product spend.


# Analyze payment methods

Find which payment methods account for the highest payment value.

In [ ]:

payment_analysis = (
    payments.groupby("payment_type", as_index=False)
    .agg(
        total_transactions=("order_id", "count"),
        total_payment_value=("payment_value", "sum"),
        average_payment_value=("payment_value", "mean")
    )
    .sort_values("total_payment_value", ascending=False)
)

display(payment_analysis)


# Visualize payment methods

In [ ]:

plt.figure(figsize=(10, 5))

sns.barplot(
    data=payment_analysis,
    x="payment_type",
    y="total_payment_value"
)

plt.title("Total Payment Value by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Total Payment Value")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


# Analyze delivery performance
We'll calculate delivery time and whether an order arrived later than its estimated delivery date.

In [ ]:

delivery = orders.loc[
    (orders["order_status"] == "delivered") &
    orders["order_purchase_timestamp"].notna() &
    orders["order_delivered_customer_date"].notna() &
    orders["order_estimated_delivery_date"].notna()
].copy()

delivery["delivery_days"] = (
    delivery["order_delivered_customer_date"]
    - delivery["order_purchase_timestamp"]
).dt.total_seconds() / 86400

delivery["delay_days"] = (
    delivery["order_delivered_customer_date"]
    - delivery["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

delivery["delivery_result"] = np.where(
    delivery["delay_days"] > 0,
    "Late",
    "On time or early"
)

print("Orders analyzed:", len(delivery))
print("Average delivery time:", round(delivery["delivery_days"].mean(), 2), "days")
print("Average delay vs estimate:", round(delivery["delay_days"].mean(), 2), "days")
print("Late deliveries:", (delivery["delay_days"] > 0).sum())
print("On-time or early deliveries:", (delivery["delay_days"] <= 0).sum())


# Plot delivery times

In [ ]:

plt.figure(figsize=(10, 5))

sns.histplot(
    data=delivery,
    x="delivery_days",
    bins=30
)

plt.title("Distribution of Delivery Times")
plt.xlabel("Days from Purchase to Delivery")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()


# Analyze customer reviews

In [ ]:

review_analysis = (
    reviews.groupby("review_score", as_index=False)
    .agg(review_count=("review_id", "count"))
    .sort_values("review_score")
)

display(review_analysis)

print("Average review score:", round(reviews["review_score"].mean(), 2))

low_review_pct = (
    reviews["review_score"].isin([1, 2]).mean() * 100
)

print("Percentage of low reviews (1–2):", round(low_review_pct, 2), "%")


# Connect delivery performance to reviews
This helps investigate whether late deliveries are associated with lower customer satisfaction.

In [ ]:

delivery_review = delivery[
    ["order_id", "delivery_result", "delay_days"]
].merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="inner",
    validate="one_to_many"
)

delivery_review_summary = (
    delivery_review.groupby("delivery_result", as_index=False)
    .agg(
        average_review_score=("review_score", "mean"),
        review_count=("review_score", "count")
    )
)

display(delivery_review_summary)


# Visualize the comparison:

In [ ]:

plt.figure(figsize=(8, 5))

sns.barplot(
    data=delivery_review_summary,
    x="delivery_result",
    y="average_review_score"
)

plt.title("Delivery Performance vs Customer Review Score")
plt.xlabel("Delivery Result")
plt.ylabel("Average Review Score")
plt.ylim(0, 5)
plt.tight_layout()
plt.show()


**The result can help identify an association between late delivery and customer satisfaction; it does not prove that delivery delays caused a particular review score.**

# **Step 7: Final KPI Summary and Exporting The Results**

# Calculate the main business KPIs

In [ ]:

# Main sales KPIs
total_revenue = sales["price"].sum()
total_orders = sales["order_id"].nunique()
total_items_sold = len(sales)
average_order_value = (
    total_revenue / total_orders if total_orders else 0
)

# Customer KPIs
total_customers = customer_orders["customer_unique_id"].nunique()
repeat_customers = (
    customer_orders["total_orders"].gt(1).sum()
)

repeat_customer_rate = (
    repeat_customers / total_customers * 100
    if total_customers else 0
)

# Delivery KPIs
late_delivery_rate = (
    delivery["delay_days"].gt(0).mean() * 100
    if len(delivery) else 0
)

# Review KPIs
average_review_score = reviews["review_score"].mean()

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Product Revenue",
        "Total Delivered Orders",
        "Total Items Sold",
        "Average Order Value",
        "Unique Customers",
        "Repeat Customers",
        "Repeat Customer Rate (%)",
        "Late Delivery Rate (%)",
        "Average Review Score"
    ],
    "Value": [
        round(total_revenue, 2),
        total_orders,
        total_items_sold,
        round(average_order_value, 2),
        total_customers,
        repeat_customers,
        round(repeat_customer_rate, 2),
        round(late_delivery_rate, 2),
        round(average_review_score, 2)
    ]
})

display(kpi_summary)


## Notebook roadmap

| Section | Purpose |
|---|---|
| 1–2 | Import libraries, load files, inspect data quality and convert dates |
| 3 | Build delivered-sales table and calculate monthly revenue and KPIs |
| 4 | Analyze revenue by product category |
| 5 | Compare one-time and repeat customers; identify high-spend customers |
| 6 | Explore payment methods, delivery performance, and review scores |
| 7 | Summarize key business KPIs and export results |

## Conclusion
Use the generated tables and charts to report the strongest revenue categories, the proportion of repeat customers, popular payment methods, delivery delays, and review-score patterns. Add specific numerical findings after running the notebook against the dataset. Delivery/review relationships are associations and should not be described as proof of causation.